In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

class VolatilityAnalyzer:
    """异常波动分析器（第二周，适配新数据集）"""
    
    def __init__(self, data_path='trump_2024_features_m2.csv'):
        self.data_path = data_path
        self.df = None
        self.anomaly_results = None
        self.event_dates = {
            '川普遇刺日': '2024-07-13',
            '最大涨幅日': None,
            '最大跌幅日': None,
            '最大波动日': None,
        }
        
    def load_data(self):
        print("=" * 80)
        print("加载数据")
        print("=" * 80)
        
        self.df = pd.read_csv(self.data_path)
        self.df['date'] = pd.to_datetime(self.df['date'])
        self.df.set_index('date', inplace=True)
        self.df.sort_index(inplace=True)
        
        # 检查必要字段
        required = ['price', 'daily_return_pct', 'rolling_volatility_7d']
        for col in required:
            if col not in self.df.columns:
                raise ValueError(f"数据集缺少必要字段：{col}")
        
        # 计算绝对收益率（用于排名）
        self.df['abs_daily_return'] = self.df['daily_return_pct'].abs()
        
        print(f"数据时间范围: {self.df.index.min().date()} → {self.df.index.max().date()}")
        print(f"总数据点数: {len(self.df)}")
        print(f"字段列表: {list(self.df.columns)}")
        return self.df
    
    def verify_data_accuracy(self, sample_size=5):
        print("\n" + "=" * 80)
        print("数据准确性验证")
        print("=" * 80)
        
        sample_dates = np.random.choice(self.df.index, sample_size, replace=False)
        results = []
        
        for date in sample_dates:
            idx = self.df.index.get_loc(date)
            if idx > 0:
                prev_price = self.df.iloc[idx-1]['price']
                cur_price = self.df.loc[date, 'price']
                manual_return = (cur_price - prev_price) / prev_price * 100
                auto_return = self.df.loc[date, 'daily_return_pct']
                return_match = np.isclose(manual_return, auto_return, rtol=1e-5)
            else:
                manual_return = None
                auto_return = self.df.loc[date, 'daily_return_pct']
                return_match = pd.isna(auto_return)
            
            # 将 numpy.datetime64 转换为 pandas Timestamp
            date_ts = pd.to_datetime(date)
            results.append({
                '日期': date_ts.strftime('%Y-%m-%d'),
                '手动日收益率': manual_return,
                '自动日收益率': auto_return,
                '匹配': return_match
            })
        
        results_df = pd.DataFrame(results)
        print(results_df.to_string(index=False))
        results_df.to_csv('data_verification.csv', index=False)
        print("\n✓ 验证结果已保存至 data_verification.csv")
        return results_df
    
    def detect_anomalies(self, sigma_threshold=2, vol_multiple=3, percentile_cutoff=10):
        """批量应用异常波动检测标准"""
        print("\n" + "=" * 80)
        print("批量异常波动检测")
        print("=" * 80)
        
        # 计算历史统计量
        hist_returns = self.df['daily_return_pct'].dropna()
        hist_std = hist_returns.std()
        hist_avg_vol = self.df['rolling_volatility_7d'].dropna().mean()
        
        anomaly_flags = []
        for date, row in self.df.iterrows():
            ret_abs = abs(row['daily_return_pct']) if not pd.isna(row['daily_return_pct']) else 0
            vol = row['rolling_volatility_7d'] if not pd.isna(row['rolling_volatility_7d']) else 0
            
            test1 = ret_abs > sigma_threshold * hist_std
            test2 = ret_abs > vol_multiple * hist_avg_vol if hist_avg_vol > 0 else False
            
            rank = (hist_returns.abs() >= ret_abs).sum()
            total = len(hist_returns)
            percentile = rank / total * 100 if total > 0 else 100
            test3 = percentile <= percentile_cutoff
            
            is_anomaly = test1 or test2 or test3
            anomaly_flags.append(is_anomaly)
        
        self.df['is_anomaly'] = anomaly_flags
        self.df['anomaly_reason'] = ''
        
        for idx, date in enumerate(self.df.index):
            ret_abs = abs(self.df.loc[date, 'daily_return_pct']) if not pd.isna(self.df.loc[date, 'daily_return_pct']) else 0
            reasons = []
            if ret_abs > sigma_threshold * hist_std:
                reasons.append(f"2σ")
            if ret_abs > vol_multiple * hist_avg_vol:
                reasons.append(f"3倍波动率")
            rank = (hist_returns.abs() >= ret_abs).sum()
            percentile = rank / len(hist_returns) * 100
            if percentile <= percentile_cutoff:
                reasons.append(f"前{percentile_cutoff}%")
            self.df.loc[date, 'anomaly_reason'] = ','.join(reasons) if reasons else ''
        
        anomaly_count = self.df['is_anomaly'].sum()
        total_count = len(self.df.dropna(subset=['daily_return_pct']))
        print(f"有效交易日: {total_count}")
        print(f"检测到异常波动天数: {anomaly_count} ({anomaly_count/total_count*100:.1f}%)")
        
        anomalies = self.df[self.df['is_anomaly']].nlargest(10, 'abs_daily_return')
        print("\n异常波动最显著的10天:")
        print(anomalies[['price', 'daily_return_pct', 'rolling_volatility_7d', 'anomaly_reason']].to_string())
        
        self.df.to_csv('volatility_anomaly_results.csv')
        print("\n✓ 异常检测结果已保存至 volatility_anomaly_results.csv")
        return self.df
    
    def identify_key_dates(self):
        """识别数据集中的关键日期（最大涨幅/跌幅/波动）"""
        if 'abs_daily_return' not in self.df.columns:
            self.df['abs_daily_return'] = self.df['daily_return_pct'].abs()
        
        max_gain_date = self.df['daily_return_pct'].idxmax()
        max_loss_date = self.df['daily_return_pct'].idxmin()
        max_vol_date = self.df['abs_daily_return'].idxmax()
        
        max_gain_date_ts = pd.to_datetime(max_gain_date)
        max_loss_date_ts = pd.to_datetime(max_loss_date)
        max_vol_date_ts = pd.to_datetime(max_vol_date)
        
        self.event_dates['最大涨幅日'] = max_gain_date_ts.strftime('%Y-%m-%d')
        self.event_dates['最大跌幅日'] = max_loss_date_ts.strftime('%Y-%m-%d')
        self.event_dates['最大波动日'] = max_vol_date_ts.strftime('%Y-%m-%d')
        
        print("\n关键日期识别:")
        print(f"最大涨幅日: {max_gain_date_ts.date()} ({self.df.loc[max_gain_date, 'daily_return_pct']:.2f}%)")
        print(f"最大跌幅日: {max_loss_date_ts.date()} ({self.df.loc[max_loss_date, 'daily_return_pct']:.2f}%)")
        print(f"最大波动日: {max_vol_date_ts.date()} ({self.df.loc[max_vol_date, 'abs_daily_return']:.2f}%)")
        return self.event_dates
    
    def multi_event_comparison(self):
        """多事件对比分析"""
        print("\n" + "=" * 80)
        print("多事件对比分析")
        print("=" * 80)
        
        if any(v is None for v in self.event_dates.values()):
            self.identify_key_dates()
        
        comparison = []
        for event_name, date_str in self.event_dates.items():
            date = pd.to_datetime(date_str)
            if date not in self.df.index:
                print(f"警告: {event_name}日期 {date_str} 不在数据中")
                continue
            
            row = self.df.loc[date]
            ret_abs = abs(row['daily_return_pct']) if not pd.isna(row['daily_return_pct']) else 0
            
            all_abs = self.df['daily_return_pct'].abs().dropna()
            rank = (all_abs >= ret_abs).sum()
            total = len(all_abs)
            percentile = rank / total * 100 if total > 0 else 0
            
            hist_avg_vol = self.df['rolling_volatility_7d'].dropna().mean()
            vol_multiple = row['rolling_volatility_7d'] / hist_avg_vol if hist_avg_vol > 0 and not pd.isna(row['rolling_volatility_7d']) else 0
            
            comparison.append({
                '事件名称': event_name,
                '日期': date_str,
                '价格': row['price'],
                '日收益率(%)': row['daily_return_pct'],
                '7日波动率': row['rolling_volatility_7d'],
                '波动率倍数': vol_multiple,
                '百分位排名(%)': percentile,
                '是否异常': '是' if row.get('is_anomaly', False) else '否',
                '异常原因': row.get('anomaly_reason', '')
            })
        
        comp_df = pd.DataFrame(comparison)
        print(comp_df.to_string(index=False))
        comp_df.to_csv('multi_event_comparison.csv', index=False)
        print("\n✓ 多事件对比结果已保存至 multi_event_comparison.csv")
        return comp_df
    
    def generate_report(self):
        """生成综合报告"""
        report_lines = []
        report_lines.append("=" * 80)
        report_lines.append("异常波动分析报告（第二周）")
        report_lines.append("=" * 80)
        report_lines.append(f"生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        report_lines.append(f"数据范围: {self.df.index.min().date()} 至 {self.df.index.max().date()}")
        report_lines.append("")
        
        anomaly_count = self.df['is_anomaly'].sum()
        total = len(self.df.dropna(subset=['daily_return_pct']))
        report_lines.append("一、异常波动总体统计")
        report_lines.append("-" * 40)
        report_lines.append(f"有效交易日: {total}")
        report_lines.append(f"异常天数: {anomaly_count} ({anomaly_count/total*100:.1f}%)")
        
        reason_counts = self.df['anomaly_reason'].value_counts().to_dict()
        report_lines.append("\n异常原因分布:")
        for reason, cnt in reason_counts.items():
            if reason:
                report_lines.append(f"  {reason}: {cnt}天")
        
        report_lines.append("\n二、关键日期对比")
        report_lines.append("-" * 40)
        comp_df = self.multi_event_comparison()
        for _, row in comp_df.iterrows():
            report_lines.append(f"{row['事件名称']} ({row['日期']}):")
            report_lines.append(f"  收益率: {row['日收益率(%)']:.2f}%")
            report_lines.append(f"  波动率倍数: {row['波动率倍数']:.2f}x")
            report_lines.append(f"  百分位排名: {row['百分位排名(%)']:.1f}%")
            report_lines.append(f"  异常判定: {row['是否异常']} ({row['异常原因']})")
            report_lines.append("")
        
        report_content = "\n".join(report_lines)
        with open('volatility_analysis_report.txt', 'w', encoding='utf-8') as f:
            f.write(report_content)
        print(report_content)
        print("\n✓ 综合报告已保存至 volatility_analysis_report.txt")
        return report_content
    
    def run(self):
        self.load_data()
        self.verify_data_accuracy()
        self.detect_anomalies()
        self.identify_key_dates()
        self.multi_event_comparison()
        self.generate_report()
        print("\n" + "=" * 80)
        print("第二周分析完成！")
        print("生成的文件列表:")
        files = ['data_verification.csv', 'volatility_anomaly_results.csv', 
                 'multi_event_comparison.csv', 'volatility_analysis_report.txt']
        for f in files:
            print(f"  - {f}")
        return True

if __name__ == "__main__":
    analyzer = VolatilityAnalyzer()
    analyzer.run()